# XGBoost Random-Split mRNA Data Investigation

This notebook uses the strongest easy-view XGBoost setup from the saved baseline notebook: random split, mRNA included, and frozen Optuna-tuned hyperparameters.

In [4]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

project_root = Path.cwd().resolve()
while not (project_root / "utils").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / "utils").exists():
    raise RuntimeError("Could not locate project root containing the utils package")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.merge_historic_data import load_merged_dataset
from utils.pipeline import SiRNADataPipeline
from utils.splitter import GroupKFoldLeakPerGroup

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Build Current Pipeline Data

This investigation uses the current shared preprocessing setup: `strict_cleaning=True`, `add_mrna=True`, and `use_normalized_conditions=False`.

Important note: the frozen XGBoost hyperparameters were tuned in the earlier baseline notebook, not re-tuned on this stricter pipeline. That is acceptable for data investigation, but the results should be treated as investigation-first rather than final model selection.

In [5]:
cmsirna_path = os.environ.get("CMSIRNA_RAW_DATA_PATH")
historic_path = os.environ.get("CMSIRNA_RAW_HISTORIC_DATA_PATH")

assert cmsirna_path, "CMSIRNA_RAW_DATA_PATH is not set"
assert historic_path, "CMSIRNA_RAW_HISTORIC_DATA_PATH is not set"

raw_df = load_merged_dataset(cmsirna_path, historic_path)
pipeline = SiRNADataPipeline(target_len=25, fetch_missing_mrna=True)
enriched_df = pipeline.enrich_dataset_with_encodings(
    raw_df,
    strict_cleaning=True,
    add_mrna=True,
)
X, groups, y = pipeline.prepare_for_classical_ml(
    enriched_df,
    target_column="Inhibition",
    use_normalized_conditions=False,
)

mask = ~np.isnan(y)
X = X[mask]
groups = groups[mask]
y = y[mask]
analysis_df = enriched_df.loc[mask].reset_index(drop=True).copy()

analysis_df["patent_group"] = analysis_df.get(
    "patent_ID", pd.Series(index=analysis_df.index, dtype=object)
).fillna("HISTORIC_OR_UNKNOWN")
analysis_df["Authorization_status"] = analysis_df.get(
    "Authorization_status", pd.Series(index=analysis_df.index, dtype=object)
).fillna("UNKNOWN")
analysis_df["Title"] = analysis_df.get(
    "Title", pd.Series(index=analysis_df.index, dtype=object)
).fillna("HISTORIC_OR_UNKNOWN")

print("Analysis dataframe shape:", analysis_df.shape)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Unique genes:", len(np.unique(groups)))


loaded 3515 historic rows
merged 43153 CMsiRNA and 3515 historic rows into 46668
Running qc and data cleaning
dropped 4233 rows for in-vivo readings
dropped 565 rows for mM readings
dropped 115 rows for unknown or unwanted cell lines
dropped 47 rows for out-of-range inhibition
dropped 1749 rows for missing or unknown concentration
dropped 796 rows for concentration > 200 nM
filled 7522 rows for missing time of administration
dropped 2198 rows with a missing or >25 nt strand
dropped 6 columns: ['Modification_locations_Sense_strand', 'Modification_locations_Antisense_strand', 'Modifications_sense_strand', 'Modifications_AntiSense_strand_3_5', 'position_Antisense_strand', 'position_Sense_strand']
Mapping mRNA structural profiles
Error reading reference dataset: [Errno 2] No such file or directory: '/home/larsena8/software/fennec/src/fennec/support_files/train_data_v1.1.0_N=27742.csv'
Loaded 47 gene sequences from local cache
Loaded 0 gene sequences from reference CSV
Building gene -> mRNA

## Frozen Hyperparameters

These are the saved Optuna hyperparameters from the earlier random-split + mRNA XGBoost notebook.

In [6]:
def make_model(frozen_params):
    return XGBRegressor(
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
        **frozen_params,
    )


def evaluate(y_true, y_pred):
    return {
        "spearman": float(spearmanr(y_true, y_pred).statistic),
        "pearson": float(pearsonr(y_true, y_pred).statistic),
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }


def grouped_spearman(df, group_cols, min_samples=10):
    rows = []
    for keys, group_df in df.groupby(group_cols, dropna=False):
        if len(group_df) < min_samples:
            continue
        corr = spearmanr(group_df["y_true"], group_df["y_pred"], nan_policy="omit").statistic
        if pd.isna(corr):
            continue
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: value for col, value in zip(group_cols, keys)}
        row.update({
            "n_samples": len(group_df),
            "spearman": float(corr),
            "mae": float(group_df["abs_error"].mean()),
            "mean_true": float(group_df["y_true"].mean()),
            "mean_pred": float(group_df["y_pred"].mean()),
        })
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["spearman", "mae"], ascending=[True, False]).reset_index(drop=True)


def issue_summary(df, group_col, min_samples=20):
    rows = []
    for value, group_df in df.groupby(group_col, dropna=False):
        if len(group_df) < min_samples:
            continue
        corr = spearmanr(group_df["y_true"], group_df["y_pred"], nan_policy="omit").statistic
        rows.append({
            group_col: value,
            "n_samples": len(group_df),
            "mae": float(group_df["abs_error"].mean()),
            "rmse": float(np.sqrt(np.mean(np.square(group_df["residual"])))),
            "spearman": float(corr) if not pd.isna(corr) else np.nan,
            "mean_true": float(group_df["y_true"].mean()),
            "mean_pred": float(group_df["y_pred"].mean()),
        })
    return pd.DataFrame(rows).sort_values(["spearman", "mae"], ascending=[True, False]).reset_index(drop=True)


## Random-Split Evaluation

This split is the easier view of the data. It is useful for asking whether suspicious subsets still behave badly even when train and test are relatively similar.

In [7]:
frozen_params = {'n_estimators': 700, 'max_depth': 9, 'learning_rate': 0.05644252612276648, 'subsample': 0.9908560353238086, 'colsample_bytree': 0.8341322370847281, 'min_child_weight': 7, 'reg_lambda': 5.683229059379634, 'reg_alpha': 4.3323727615796335, 'gamma': 0.6436988616639515}
model = make_model(frozen_params)

row_idx = np.arange(len(analysis_df))
X_train, X_test, y_train, y_test, groups_train, groups_test, idx_train, idx_test = train_test_split(
    X,
    y,
    groups,
    row_idx,
    test_size=0.33,
    random_state=42,
)

model.fit(X_train, y_train)
y_test_pred = model.predict(X_test)

split_summary = pd.DataFrame([{
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "n_train_groups": int(len(np.unique(groups_train))),
    "n_test_groups": int(len(np.unique(groups_test))),
}])
metrics_df = pd.DataFrame([evaluate(y_test, y_test_pred)])

predictions_df = analysis_df.iloc[idx_test].reset_index().rename(columns={"index": "source_index"}).copy()
predictions_df["row_index"] = idx_test
predictions_df["group"] = groups_test
predictions_df["y_true"] = y_test
predictions_df["y_pred"] = y_test_pred
predictions_df["residual"] = predictions_df["y_true"] - predictions_df["y_pred"]
predictions_df["abs_error"] = predictions_df["residual"].abs()

split_summary


,n_train,n_test,n_train_groups,n_test_groups
0,23747,11697,54,54


## Split Summary

## Overall Metrics

In [8]:
metrics_df

,spearman,pearson,rmse,mae
0,0.862976,0.85009,17.557462,12.971708


## Concentration Ranking Slice

These tables show where XGBoost preserves ranking poorly or well across concentration alone and the key concentration combinations.

In [9]:
spearman_by_concentration = grouped_spearman(predictions_df, ["Concentration_nM"], min_samples=20)
spearman_by_concentration_gene = grouped_spearman(predictions_df, ["Concentration_nM", "group"], min_samples=10)
spearman_by_concentration_patent = grouped_spearman(predictions_df, ["Concentration_nM", "patent_group"], min_samples=10)

spearman_by_concentration.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,n_samples,spearman,mae,mean_true,mean_pred
0,0.00300,23,0.326168,10.918688,4.235217,5.176704
1,0.00064,38,0.365927,8.440850,-1.724211,-5.312652
2,150.00000,22,0.404856,16.473268,43.615455,31.908409
3,0.01000,65,0.606163,13.066145,38.883077,36.976765
4,200.00000,40,0.626643,13.035901,76.792750,67.531006
5,20.00000,124,0.655586,18.173071,42.357903,48.099728
6,40.00000,22,0.660447,14.167580,34.350000,34.392075
7,0.00320,35,0.686935,7.900771,1.807429,2.786147
8,0.00100,24,0.697391,9.568274,13.613750,8.447338
9,0.50000,227,0.708774,11.638138,16.396784,14.155383


## Concentration + Gene Hotspots

In [10]:
spearman_by_concentration_gene.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,group,n_samples,spearman,mae,mean_true,mean_pred
0,50.00000,UBE2J1,13,-0.219780,15.509253,63.746154,71.038803
1,0.00300,LPA,10,-0.200000,14.923315,-2.167000,0.270676
2,2.00000,LPA,16,-0.020588,22.869775,9.150000,10.907495
3,0.20000,LPA,10,0.018182,19.323079,-10.930000,-18.908451
4,100.00000,ICAM-1,22,0.088801,20.664006,27.772727,40.055309
5,20.00000,PNPLA3,43,0.127303,17.629635,22.950930,32.062481
6,0.00064,AGT,34,0.206015,8.206746,-3.870882,-7.404373
7,50.00000,P2RX2,22,0.291925,12.372729,60.118182,67.683182
8,100.00000,FIREFLY LUC,35,0.341552,18.420919,70.460419,69.535660
9,150.00000,HSD17B13,22,0.404856,16.473268,43.615455,31.908409


## Concentration + Patent Hotspots

In [11]:
spearman_by_concentration_patent.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,patent_group,n_samples,spearman,mae,mean_true,mean_pred
0,0.00300,WO2023109940A1,10,-0.200000,14.923315,-2.167000,0.270676
1,10.00000,CN112313335A,52,-0.082750,6.450536,92.330385,89.436607
2,0.10000,WO2022121959A1,21,-0.064390,44.014354,61.776667,49.911209
3,2.00000,CN108368506A,16,-0.020588,22.869775,9.150000,10.907495
4,0.10000,CN112313335A,56,0.000786,9.401336,98.690893,92.707718
5,0.20000,CN108368506A,10,0.018182,19.323079,-10.930000,-18.908451
6,10.00000,WO2022121959A1,10,0.103344,25.381388,67.011000,75.591293
7,0.00064,WO2023088427A1,25,0.113922,9.245764,-2.142400,-7.310833
8,20.00000,US20220117999A1,43,0.127303,17.629635,22.950930,32.062481
9,5.00000,WO2023091644A2,13,0.202223,12.664140,21.062308,23.009672


## Experimental Feature Issues

These summaries highlight which experimental contexts are associated with poor ranking or high error in this XGBoost setup.

In [12]:
issue_by_cell_type = issue_summary(predictions_df, "Cell_Type", min_samples=30)
issue_by_concentration = issue_summary(predictions_df, "Concentration_nM", min_samples=20)
issue_by_time = issue_summary(predictions_df, "Time_of_administration_h", min_samples=20)
issue_by_patent = issue_summary(predictions_df, "patent_group", min_samples=20)
issue_by_authorization = issue_summary(predictions_df, "Authorization_status", min_samples=20)

issue_by_cell_type.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Cell_Type,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,T24 Cells,38,16.917073,22.083472,0.318768,30.052632,40.795864
1,HEK293 Cells,99,15.786432,21.147632,0.570972,66.506727,66.624252
2,Non-human hepatocytes,61,9.764392,12.463616,0.580015,25.152459,24.400068
3,H1299 Cells,813,12.032245,14.936538,0.620351,68.100000,67.960320
4,Primary human hepatocytes,945,15.627904,22.112292,0.658753,52.506593,52.114281
5,HeLa Cells,245,12.510657,15.758988,0.741065,54.823783,57.137627
6,HepG2,564,14.075676,19.159366,0.769334,41.148298,41.533348
7,COS7,1272,12.330233,15.827911,0.796480,25.088522,24.489094
8,Huh7,529,13.021323,19.002876,0.809707,40.530038,40.021416
9,Human iPSC-derived cortical neurons,66,13.141103,19.311495,0.813444,46.122727,42.585159


## Concentration Issue Summary

In [13]:
issue_by_concentration.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,0.00300,23,10.918688,13.826063,0.326168,4.235217,5.176704
1,0.00064,38,8.440850,11.874089,0.365927,-1.724211,-5.312652
2,150.00000,22,16.473268,21.509233,0.404856,43.615455,31.908409
3,0.01000,65,13.066145,16.734926,0.606163,38.883077,36.976765
4,200.00000,40,13.035901,20.671536,0.626643,76.792750,67.531006
5,20.00000,124,18.173071,23.311226,0.655586,42.357903,48.099728
6,40.00000,22,14.167580,18.754924,0.660447,34.350000,34.392075
7,0.00320,35,7.900771,10.185518,0.686935,1.807429,2.786147
8,0.00100,24,9.568274,13.601906,0.697391,13.613750,8.447338
9,0.50000,227,11.638138,14.163890,0.708774,16.396784,14.155383


## Time Issue Summary

In [14]:
issue_by_time.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Time_of_administration_h,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,4.0,38,16.917073,22.083472,0.318768,30.052632,40.795864
1,40.0,60,9.697118,13.546927,0.730607,29.293333,27.041281
2,72.0,489,10.839710,13.645297,0.761562,17.052863,15.416394
3,168.0,66,13.141103,19.311495,0.813444,46.122727,42.585159
4,48.0,3014,12.347541,15.980241,0.815273,51.194126,51.276035
5,24.0,8030,13.340221,18.305844,0.865901,41.820575,41.916813


## Patent Issue Summary

In [15]:
issue_by_patent.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,patent_group,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,WO2022121959A1,31,38.003720,49.906056,-0.078273,63.465161,58.195103
1,CN112313335A,108,7.980580,11.233991,0.181565,95.628426,91.132751
2,WO2023056446A1,77,16.809603,20.764039,0.394104,63.428571,59.636662
3,US20220380773A1,51,25.053285,27.943498,0.410702,37.431373,44.217323
4,CN116801886A,91,18.531338,23.225768,0.480213,49.835275,51.676689
5,CN108368506A,83,16.940998,21.357671,0.556298,12.746988,10.254803
6,WO2022089486A1,87,8.983158,10.805960,0.573729,77.655172,76.235977
7,TW202321444A,61,9.764392,12.463616,0.580015,25.152459,24.400068
8,CN117070517A,29,7.387632,10.222228,0.627094,66.277241,63.498779
9,CN116515835A,127,10.538042,12.994963,0.638101,62.557953,62.085636


## Authorization Issue Summary

In [16]:
issue_by_authorization.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Authorization_status,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,Published,70,9.961700,14.001279,0.672657,26.672857,26.247963
1,UNKNOWN,1195,12.596680,15.976761,0.685246,64.036228,64.766945
2,Withdrawn,366,15.208136,18.700145,0.787707,28.931967,29.910517
3,Unknown Status,4212,12.880727,17.602940,0.832576,33.914421,34.216629
4,Granted,580,14.041393,18.451008,0.851171,49.670310,50.424637
5,Substantive Examination,5274,12.896457,17.722134,0.871723,46.220146,45.717113


## Prediction Preview

In [17]:
predictions_df[["group", "Cell_Type", "Concentration_nM", "Time_of_administration_h", "patent_group", "y_true", "y_pred", "residual", "abs_error"]].head(20)

,group,Cell_Type,Concentration_nM,Time_of_administration_h,patent_group,y_true,y_pred,residual,abs_error
0,PNPLA3,COS7,1.00,24.0,CN115176007A,32.00,19.588490,12.411510,12.411510
1,PNPLA3,HepG2,10.00,24.0,WO2022256395A1,38.80,40.274586,-1.474586,1.474586
2,PCSK9,HepG2,30.00,24.0,CN101484588B,50.00,39.119659,10.880341,10.880341
3,PNPLA3,Hep3B,10.00,24.0,WO2022256395A1,56.48,44.831028,11.648972,11.648972
4,MMAC1,T24 Cells,100.00,4.0,HISTORIC_OR_UNKNOWN,30.00,38.653091,-8.653091,8.653091
5,HSD17B13,COS7,50.00,48.0,CN112020556A,52.00,51.185616,0.814384,0.814384
6,APP,Be(2)C cell line,10.00,24.0,WO2020132227A2,69.07,55.956715,13.113285,13.113285
7,AGT,Hep3B,10.00,24.0,CN106574268B,80.79,74.063721,6.726279,6.726279
8,PCSK9,Hela,100.00,48.0,CN113980966A,79.00,68.757095,10.242905,10.242905
9,HSD17B13,Huh7,0.05,24.0,WO2023220561A1,38.83,36.150669,2.679331,2.679331


## Interpretation Notes

- Overall, this easy-view XGBoost setup performs very strongly: Spearman is about `0.863`, Pearson about `0.850`, RMSE about `17.56`, and MAE about `12.97`.
- The main interpretation is that the current dataset is clearly learnable when train and test are allowed to be similar. That means we are not looking at a dataset that is globally broken or impossible to model.
- Because this is a random split, these results should be used mainly to detect subsets that still behave badly even in an easy setting. Those are stronger candidates for real label inconsistency or protocol noise.
- Concentration-only behavior is mostly strong here. Even `0.1 nM`, `0.5 nM`, and `1.0 nM` look reasonably learnable in this easier split, which suggests that low dose alone is not automatically poisoning the dataset.
- The weakest concentration slices are mostly very small-sample ultra-low-dose regimes such as `0.003`, `0.00064`, and `0.010 nM`. Those look more like rare or unstable assay regimes than broad global problems.
- In the `Concentration + Gene` view, the worst slices are mostly small groups such as `UBE2J1 @ 50 nM`, `LPA @ 0.003/0.2/2.0 nM`, and `ICAM-1 @ 100 nM`. These deserve review, but many are too small to justify broad deletion by themselves.
- In the `Concentration + Patent` view, a few patent hotspots still look suspicious even under random split, especially `WO2022121959A1 @ 0.1 nM`, `CN112313335A @ 10.0 nM`, and `US20220117999A1 @ 20.0 nM`. These are stronger review candidates because they remain weak even in an easier modeling setup.
- Experimental feature summaries are generally reassuring in this notebook. Most cell types and time points show moderate-to-strong Spearman. `Primary mouse hepatocytes` no longer looks catastrophic here, which suggests that its main problem may be transfer/generalization rather than pure label noise.
- Patent-level summaries are more concerning than cell-type summaries in this notebook. `WO2022121959A1` is the clearest patent-level warning sign because its overall patent Spearman is negative even in the random-split setup.

## Conclusions And Next Step

- This notebook says the dataset is broadly learnable and not globally poisoned.
- The easiest-view warning signs are more concentrated around specific patent hotspots than around broad concentration ranges.
- The by-gene notebook should therefore be treated as the main decision-maker for filtering before deep learning.
- The best next step is to focus on the overlap between the RF notebooks and the XGBoost by-gene notebook. Subsets that look bad in both are the strongest poisoning candidates.
- Based on this notebook alone, I would **not** remove all low-dose rows. I would instead prioritize targeted review of suspicious patent slices such as `WO2022121959A1`, `CN112313335A`, and `US20220117999A1` in their weak concentration bands.
